# <font color = 'indianred'>**Multilabel Classification of StackExchange Dataset using GEMMA** </font>

**Objective:**

In this notebook, we aim to use GEMMA models with QLORA for classification problems. **We will now use Casual Languagge Model - Basically we will do instruction tuning.**


**Plan**

1. Set Environment
2. Load Dataset
3. Accessing and Manipulating Splits
4. Load Pre-trained Tokenizer
5. Create Prompts
6. Model Training
  1. Download pre-trained model <br>  
  3. PEFT Setup
  4. Training Arguments <br>
  5. Instantiate Trainer <br>
  6. Setup WandB <br>
  7. Training























# <font color = 'indianred'> **1. Setting up the Environment** </font>



In [2]:

# If in Colab, then import the drive module from google.colab
if 'google.colab' in str(get_ipython()):
  #!pip install uv
  !pip install numpy -U -qq
  !pip install transformers evaluate wandb datasets accelerate trl peft bitsandbytes -U -qq

 <Font size = 5 color = 'indianred'>**Restart the session before moving onto next cell**
> Runtime- Restart Session

<font color = 'indianred'> *Load Libraries* </font>

In [2]:
# standard python libraries
from pathlib import Path
from typing import Dict, List, Union, Optional, Tuple
from tqdm import tqdm
import json
import joblib
import os
import sys

# Data Science librraies
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import multilabel_confusion_matrix, precision_score, recall_score, f1_score

# Pytorch
import torch
import torch.nn as nn

# Huggingface Librraies
import evaluate
from datasets import load_dataset, DatasetDict, Dataset, ClassLabel
from trl import SFTConfig, SFTTrainer
from transformers import (
    set_seed,
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    AutoConfig,
    BitsAndBytesConfig,
)
from peft import (
    TaskType,
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
    AutoPeftModelForCausalLM,
    PeftConfig
)
# Logging and secrets
from huggingface_hub import login, HfApi, create_repo
from google.colab import userdata
import wandb

In [3]:
sys.path

['/content',
 '/env/python',
 '/usr/lib/python312.zip',
 '/usr/lib/python3.12',
 '/usr/lib/python3.12/lib-dynload',
 '',
 '/usr/local/lib/python3.12/dist-packages',
 '/usr/lib/python3/dist-packages',
 '/usr/local/lib/python3.12/dist-packages/IPython/extensions',
 '/root/.ipython',
 '/tmp/tmpk5kcoyyo']

In [10]:
# If running on Google Colab, use Google Drive as storage
# CHANGE FOLDERS TO WHERE YOU WANT TO SAVE DATA AND MODELS

if 'google.colab' in str(get_ipython()):
    from google.colab import drive  # Import Google Drive mounting utility
    drive.mount('/content/drive')  # Mount Google Drive

    # Set base folder path for storing data on Google Drive
    data_folder= Path('/content/drive/MyDrive')
    project_folder = Path('/content/drive/MyDrive')
    base_folder = Path('/content/drive/MyDrive')

# If running locally, specify a different path
else:
    # Set base folder path for storing data on local machine
    data_folder= Path('/home/harpreet/Insync/google_drive_shaannoor/data')
    project_folder= Path('/home/harpreet/Insync/google_drive_shaannoor/teaching_fall_2025/LLM_Fall_2025/')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
util_folder = project_folder/'shared_utils'
util_folder

PosixPath('/content/drive/MyDrive/shared_utils')

In [12]:
sys.path.append(str(util_folder))

In [13]:
sys.path

['/content',
 '/env/python',
 '/usr/lib/python312.zip',
 '/usr/lib/python3.12',
 '/usr/lib/python3.12/lib-dynload',
 '',
 '/usr/local/lib/python3.12/dist-packages',
 '/usr/lib/python3/dist-packages',
 '/usr/local/lib/python3.12/dist-packages/IPython/extensions',
 '/root/.ipython',
 '/tmp/tmpk5kcoyyo',
 '/content/drive/MyDrive/shared_utils',
 '/content/drive/MyDrive/shared_utils']

In [14]:
print("Exists:", util_folder.exists())

Exists: True


In [15]:
from shared_utils import free_gpu_memory, find_linear_layers, multilabel_evaluation, get_appropriate_dtype

In [16]:
wandb_api_key = userdata.get('WANDB_API_KEY')
hf_token = userdata.get('HF_TOKEN')

In [17]:
if hf_token:
    # Log in to Hugging Face
    login(token=hf_token)
    print("Successfully logged in to Hugging Face!")
else:
    print("Hugging Face token not found in notebook secrets.")

Successfully logged in to Hugging Face!


In [18]:
if wandb_api_key:
  wandb.login(key=wandb_api_key)
  print("Successfully logged in to WANDB!")
else:
    print("WANDB key not found in notebook secrets.")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: younes-hoseini67 (younes-hoseini67-university-of-texas-at-dallas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged in to WANDB!


In [21]:

kaggle_api = base_folder/'.kaggle'
import os
if 'google.colab' in str(get_ipython()):
    os.environ['KAGGLE_CONFIG_DIR'] = "/content/drive/MyDrive/.kaggle/"
if 'google.colab' in str(get_ipython()):
    !chmod 600 /content/drive/MyDrive/.kaggle/kaggle.json
if 'google.colab' in str(get_ipython()):
    ! ls -la  /content/drive/MyDrive/.kaggle/kaggle.json


-rw------- 1 root root 67 Oct 27 23:03 /content/drive/MyDrive/.kaggle/kaggle.json


# <font color = 'indianred'> **2. Load Data set**
    


In [22]:
!kaggle competitions download -c emotion-detection-fall-2025-dl
!unzip -o emotion-detection-fall-2025-dl.zip -d {data_folder}

  0% 0.00/609k [00:00<?, ?B/s]
100% 609k/609k [00:00<00:00, 1.09GB/s]
Archive:  emotion-detection-fall-2025-dl.zip
  inflating: /content/drive/MyDrive/sample_submission.csv  
  inflating: /content/drive/MyDrive/test.csv  
  inflating: /content/drive/MyDrive/train.csv  


In [23]:
train = pd.read_csv(data_folder/'train.csv')
print(f"Kaggle train data shape: {train.shape}")
print(f"Columns: {train.columns.tolist()}")

# Use the Kaggle training data
df = train
df

Kaggle train data shape: (7724, 13)
Columns: ['ID', 'Tweet', 'anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']


,ID,Tweet,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust
0,2017-21441,“Worry is a down payment on a problem you may ...,0,1,0,0,0,0,1,0,0,0,1
1,2017-31535,Whatever you decide to do make sure it makes y...,0,0,0,0,1,1,1,0,0,0,0
2,2017-21068,@Max_Kellerman it also helps that the majorit...,1,0,1,0,1,0,1,0,0,0,0
3,2017-31436,Accept the challenges so that you can literall...,0,0,0,0,1,0,1,0,0,0,0
4,2017-22195,My roommate: it's okay that we can't spell bec...,1,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7719,2018-01993,@BadHombreNPS @SecretaryPerry If this didn't m...,1,0,1,0,0,0,0,0,0,0,0
7720,2018-01784,Excited to watch #stateoforigin tonight! Come ...,0,0,0,0,1,0,1,0,0,0,0
7721,2018-04047,"Blah blah blah Kyrie, IT, etc. @CJC9BOSS leavi...",1,0,1,0,0,0,0,0,1,0,0
7722,2018-03041,#ThingsIveLearned The wise #shepherd never tru...,0,0,0,0,0,0,0,0,0,0,0


In [24]:
class_names = ['anger', 'anticipation', 'disgust', 'fear', 'joy',
               'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']

# Convert binary columns to list of emotion labels
def get_emotion_labels(row):
    emotions = [emotion for emotion in class_names if row[emotion] == 1]
    return json.dumps(emotions)

df['label'] = df.apply(get_emotion_labels, axis=1)

# Create final dataframe with text and label
df_final = df[['Tweet', 'label']].rename(columns={'Tweet': 'text'})

print(f"\nFinal dataset shape: {df_final.shape}")
print(f"Sample label: {df_final['label'][0]}")


Final dataset shape: (7724, 2)
Sample label: ["anticipation", "optimism", "trust"]


In [25]:
emotion_dataset_final = Dataset.from_pandas(df_final)
emotion_dataset_final[0]['label']

'["anticipation", "optimism", "trust"]'

# <font color = 'indianred'> **3. Accessing and Manuplating Splits**</font>



<font color = 'indianred'>*Create futher subdivions of the splits*</font>

In [26]:
# Split the test set into test and validation sets
test_val_splits = emotion_dataset_final.train_test_split(test_size=0.4, seed=42)
train_split= test_val_splits['train']
test_val_splits = test_val_splits['test'].train_test_split(test_size=0.5, seed=42,)
val_split = test_val_splits['train']
test_split = test_val_splits['test']


<font color = 'indianred'>*small subset for initial experimenttaion*</font>

In [27]:
train_split = train_split.shuffle(seed = 42).select(range(min(2000, len(train_split))))
val_split = val_split.shuffle(seed = 42).select(range(min(2000, len(val_split))))
test_split = test_split.shuffle(seed = 42).select(range(min(2000, len(test_split))))


In [28]:
data_subset = DatasetDict({"train": train_split, "valid": val_split, "test": test_split})

In [29]:
data_subset['train'][0]

{'text': 'Dear future big brother players, just chase dick all season and you too can win 500,00 dollars and an Sti #bb18 ',
 'label': '["anger", "joy", "optimism"]'}

# <font color = 'indianred'>**4. Load pre-trained Tokenizer**</font>



In [30]:
free_gpu_memory()

GPU memory has been freed.


In [31]:
checkpoint = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [32]:
tokenizer.eos_token

'<eos>'

In [33]:
tokenizer.pad_token

'<pad>'

In [34]:
tokenizer.padding_side

'left'

In [35]:
tokenizer.chat_template

#<font color = 'indianred'> **5. Create Completion Dataset**



In [36]:
class_names

['anger',
 'anticipation',
 'disgust',
 'fear',
 'joy',
 'love',
 'optimism',
 'pessimism',
 'sadness',
 'surprise',
 'trust']

In [37]:
def format_prompt_completion(example):
    prompt = f"Classify the TEXT by selecting all applicable labels from the following list: {class_names}. ### TEXT: {example['text'].strip()} ### LABEL:"
    completion = f" {example['label'].strip()}"
    return {"prompt": prompt, "completion": completion}


In [38]:
data_subset_completition = data_subset.map(format_prompt_completion, remove_columns=["text", "label"])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1545 [00:00<?, ? examples/s]

Map:   0%|          | 0/1545 [00:00<?, ? examples/s]

In [39]:
data_subset_completition

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 2000
    })
    valid: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1545
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1545
    })
})

In [40]:
data_subset_completition['train'][0]

{'prompt': "Classify the TEXT by selecting all applicable labels from the following list: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']. ### TEXT: Dear future big brother players, just chase dick all season and you too can win 500,00 dollars and an Sti #bb18 ### LABEL:",
 'completion': ' ["anger", "joy", "optimism"]'}

##  <font color = 'indianred'> **5.1 Filter Longer sequences**

In [41]:
data_subset_completition = data_subset_completition.map(
    lambda example: {'keep': len(tokenizer.encode(example['prompt'])) <= 500}
)

# Then filter
data_subset_completition = data_subset_completition.filter(lambda x: x['keep'])
data_subset_completition = data_subset_completition.remove_columns(['keep'])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1545 [00:00<?, ? examples/s]

Map:   0%|          | 0/1545 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1545 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1545 [00:00<?, ? examples/s]

In [42]:
data_subset_completition

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 2000
    })
    valid: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1545
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1545
    })
})

In [43]:
data_subset_completition['train'][0]

{'prompt': "Classify the TEXT by selecting all applicable labels from the following list: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']. ### TEXT: Dear future big brother players, just chase dick all season and you too can win 500,00 dollars and an Sti #bb18 ### LABEL:",
 'completion': ' ["anger", "joy", "optimism"]'}

##  <font color = 'indianred'> **5.2 Push Dataset to Hub**

In [44]:
data_subset_completition.push_to_hub(
    "sayedyounes/stack_multilabel_subset_completion",
    private=False  # Set to True if you want it private
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  92%|#########2|  181kB /  195kB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  24%|##4       | 37.2kB /  153kB            

README.md:   0%|          | 0.00/512 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/sayedyounes/stack_multilabel_subset_completion/commit/055929a1706207c9e54f1fdf5ab4a6fcab790b5f', commit_message='Upload dataset', commit_description='', oid='055929a1706207c9e54f1fdf5ab4a6fcab790b5f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/sayedyounes/stack_multilabel_subset_completion', endpoint='https://huggingface.co', repo_type='dataset', repo_id='sayedyounes/stack_multilabel_subset_completion'), pr_revision=None, pr_num=None)

In [45]:
train_filtered = data_subset_completition['train']
valid_filtered = data_subset_completition['valid']


#  <font color = 'indianred'> **6. Model Training**

##  <font color = 'indianred'> **6.1 Download pre-trained model**

In [46]:
torch_data_type = get_appropriate_dtype()
torch_data_type

torch.bfloat16

In [47]:
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_use_double_quant=True,
  bnb_4bit_compute_dtype=torch_data_type,
  bnb_4bit_quant_storage=torch_data_type,
)

In [48]:
model = AutoModelForCausalLM.from_pretrained(checkpoint,
                                             quantization_config=bnb_config,
                                             torch_dtype=torch_data_type,
                                             trust_remote_code=True,)


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/481M [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

##  <font color = 'indianred'> **6.2 PEFT Setup**

In [49]:
model

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear4bit(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedfor

In [50]:
find_linear_layers(model)

['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'lm_head']


['k_proj',
 'gate_proj',
 'up_proj',
 'v_proj',
 'down_proj',
 'o_proj',
 'q_proj',
 'lm_head']

In [51]:
TaskType.CAUSAL_LM

<TaskType.CAUSAL_LM: 'CAUSAL_LM'>

In [52]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=128,
    lora_alpha=256,
    lora_dropout=0.01,
    target_modules = ['v_proj',  'q_proj',  'up_proj', 'o_proj', 'down_proj', 'gate_proj','k_proj'])


## <font color = 'indianred'> **6.3 Training Arguments**</font>







In [53]:
# Define the directory where model checkpoints will be saved

model_folder = Path("/content/models/gemma_qlora_lmh")
# Create the directory if it doesn't exist
model_folder.mkdir(exist_ok=True, parents=True)
run_name= 'stack_exp_lmh_gemma_base'

use_fp16 = torch_data_type == torch.float16
use_bf16 = torch_data_type == torch.bfloat16

# Configure training parameters
training_args = SFTConfig(
    seed = 42,
    dataset_text_field="text",
    max_length = 1024,
    packing = False,
    completion_only_loss=True,

    # Training-specific configurations
    num_train_epochs=2,  # Total number of training epochs
    per_device_train_batch_size=16, # Number of samples per training batch for each device
    per_device_eval_batch_size=16,  # Number of samples per evaluation batch for each device
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant":False},
    torch_empty_cache_steps=20,
    weight_decay=0.0,  # Apply L2 regularization to prevent overfitting
    learning_rate=1e-5,  # Step size for the optimizer during training
    optim='adamw_torch',  # Optimizer,

    # Checkpoint saving and model evaluation settings
    output_dir=str(model_folder),  # Directory to save model checkpoints
    eval_strategy='steps',  # Evaluate model at specified step intervals
    eval_steps=20,  # Perform evaluation every 10 training steps
    save_strategy="steps",  # Save model checkpoint at specified step intervals
    save_steps=20,  # Save a model checkpoint every 10 training steps
    load_best_model_at_end=True,  # Reload the best model at the end of training
    save_total_limit=2,  # Retain only the best and the most recent model checkpoints
    # Use 'accuracy' as the metric to determine the best model
    metric_for_best_model="eval_loss",
    greater_is_better=False,  # A model is 'better' if its accuracy is higher


    # Experiment logging configurations (commented out in this example)
    logging_strategy='steps',
    logging_steps=20,
    report_to='wandb',  # Log metrics and results to Weights & Biases platform
    run_name= run_name,  # Experiment name for Weights & Biases

    # Precision settings determined based on GPU capability
    fp16=use_fp16 ,  # Set True if torch_data_type is torch.float16
    bf16=use_bf16,  # Set True if torch_data_type is torch.bfloat16
    tf32=False,  # Disable tf32 unless you want to use Ampere specific optimization
)


In [54]:
# If gradient checkpointing is enabled, configure relevant settings
if training_args.gradient_checkpointing:
    model.config.use_cache = False  # Disable caching for compatibility


##  <font color = 'indianred'> **6.4 Initialize Trainer**</font>



In [55]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_filtered,
    eval_dataset=valid_filtered,
    peft_config=peft_config,

)

Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1545 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1545 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1545 [00:00<?, ? examples/s]

In [56]:
dataloader = trainer.get_train_dataloader()
batch = next(iter(dataloader))

In [57]:
print(batch['input_ids'][0][0:5])
print(tokenizer.decode(batch['input_ids'][0][0:5]))
print(batch['labels'][0][0:5])

tensor([     2, 212107,    573,   1837,    731], device='cuda:0')
<bos>Classify the TEXT by
tensor([-100, -100, -100, -100, -100], device='cuda:0')


In [58]:
batch

{'input_ids': tensor([[     2, 212107,    573,  ...,   4437,      1,      0],
         [     2, 212107,    573,  ...,      0,      0,      0],
         [     2, 212107,    573,  ...,      0,      0,      0],
         ...,
         [     2, 212107,    573,  ...,      0,      0,      0],
         [     2, 212107,    573,  ...,      0,      0,      0],
         [     2, 212107,    573,  ...,      0,      0,      0]],
        device='cuda:0'),
 'labels': tensor([[-100, -100, -100,  ..., 4437,    1, -100],
         [-100, -100, -100,  ..., -100, -100, -100],
         [-100, -100, -100,  ..., -100, -100, -100],
         ...,
         [-100, -100, -100,  ..., -100, -100, -100],
         [-100, -100, -100,  ..., -100, -100, -100],
         [-100, -100, -100,  ..., -100, -100, -100]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ...

In [59]:
print(len(batch['input_ids'][0]))
print(len(batch['labels'][0]))

107
107


In [60]:
print(batch['input_ids'][0][0:5])
print(tokenizer.decode(batch['input_ids'][0][0:5]))
print(batch['labels'][0][0:5])

tensor([     2, 212107,    573,   1837,    731], device='cuda:0')
<bos>Classify the TEXT by
tensor([-100, -100, -100, -100, -100], device='cuda:0')


In [61]:
print(f"\nINPUTS")
print(f"{'-'*80}")
print(batch['input_ids'][0][99:114])
print(f"\nLABELS")
print(f"{'-'*80}")
print(batch['labels'][0][99:114])
print(f"\nTokens")
print(f"{'-'*80}")
print(tokenizer.decode(batch['input_ids'][0][99:114]))


INPUTS
--------------------------------------------------------------------------------
tensor([66047,   824,   664, 37968,  1746,  4437,     1,     0],
       device='cuda:0')

LABELS
--------------------------------------------------------------------------------
tensor([66047,   824,   664, 37968,  1746,  4437,     1,  -100],
       device='cuda:0')

Tokens
--------------------------------------------------------------------------------
mism", "sadness"]<eos><pad>


In [62]:
def verify_loss_masking(tokenizer, trainer, num_samples=3):
    """
    Verify which tokens contribute to loss (labels != -100)
    for a few samples from the training dataloader.
    """
    dataloader = trainer.get_train_dataloader()
    batch = next(iter(dataloader))

    for i in range(min(num_samples, len(batch["input_ids"]))):
        input_ids = batch["input_ids"][i]
        labels = batch["labels"][i]

        print(f"\n{'='*80}")
        print(f"Sample {i+1}")
        print(f"{'='*80}")

        # Decode full sequence for reference
        full_text = tokenizer.decode(input_ids, skip_special_tokens=False)
        print(f"\nFull text:\n{full_text}")

        # Identify tokens used for loss
        loss_token_indices = (labels != -100).nonzero(as_tuple=True)[0]

        if len(loss_token_indices) == 0:
            print("All tokens masked — no loss will be calculated.")
            continue

        print(f"\nTokens contributing to loss ({len(loss_token_indices)} total):")
        print(f"{'-'*80}")
        print(f"{'Index':<8} {'Token ID':<10} {'Token Text'}")
        print(f"{'-'*80}")

        for idx in loss_token_indices.tolist():
            token_id = input_ids[idx].item()
            token_text = tokenizer.decode([token_id], skip_special_tokens=False)
            print(f"{idx:<8} {token_id:<10} {repr(token_text)}")

        print(f"{'-'*80}")
        print(f"Percentage of tokens used for loss: {len(loss_token_indices)/len(labels)*100:.2f}%")



In [63]:
# Call this after creating your trainer
verify_loss_masking(tokenizer, trainer, num_samples=2)



Sample 1

Full text:
<bos>Classify the TEXT by selecting all applicable labels from the following list: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']. ### TEXT: @LindaFrum sadly, at least 3 more years of this. At what point will even the low info selfie lovers who voted for them say enough is enough ### LABEL: ["disgust", "pessimism", "sadness"]<eos><pad>

Tokens contributing to loss (14 total):
--------------------------------------------------------------------------------
Index    Token ID   Token Text
--------------------------------------------------------------------------------
92       10890      ' ["'
93       1999       'dis'
94       15819      'gust'
95       824        '",'
96       664        ' "'
97       3462       'pes'
98       542        'si'
99       66047      'mism'
100      824        '",'
101      664        ' "'
102      37968      'sad'
103      1746       'ness'
104      4437       '"]'
1

## <font color = 'indianred'> **6.5 Setup WandB**</font>

In [64]:
%env WANDB_PROJECT = emotion-detection

env: WANDB_PROJECT=emotion-detection


##  <font color = 'indianred'> **6.6 Start Training**

In [65]:
try:
    # Your code that may cause a CUDA out-of-memory error
    # Example: trainer.train() or other GPU intensive operations
    # lora_model.config.use_cache = False
    trainer.train()
except RuntimeError as e:
    if 'CUDA out of memory' in str(e):
        print("CUDA out of memory error detected. Freeing GPU memory.")
        free_gpu_memory()
        # Optionally, you can retry the operation here after freeing up memory
        # Example retry:
        # trainer.train()
    else:
        raise e


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
20,0.540600,0.402337,1.902760,59654.000000,0.852763
40,0.375900,0.381454,1.795316,118865.000000,0.857785
60,0.367700,0.364805,1.792044,178998.000000,0.860767
80,0.337900,0.359027,1.718429,237097.000000,0.866840
100,0.308500,0.349296,1.768806,296728.000000,0.868905
120,0.314200,0.346163,1.750096,356450.000000,0.869618


##  <font color = 'indianred'> **6.7 Push best checkpoint to Hub**

In [66]:
best_model_checkpoint_step = trainer.state.best_model_checkpoint.split('-')[-1]

In [67]:
best_model_checkpoint_step

'120'

In [68]:
checkpoint = str(model_folder/f'checkpoint-{best_model_checkpoint_step}')
checkpoint

'/content/models/gemma_qlora_lmh/checkpoint-120'

In [69]:
# Step 1: Create the repository
repo_id="sayedyounes/stack_exc_multilabel_base_lm_head"
create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=False,  # Set to True if you want it private
    exist_ok=True   # Won't error if repo already exists
)

# Step 2: Upload the folder
api = HfApi()
api.upload_folder(
    folder_path=checkpoint,
    repo_id=repo_id,
    repo_type="model",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...point-120/tokenizer.model:   0%|          | 16.2kB / 4.24MB            

  ...ckpoint-120/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...adapter_model.safetensors:   0%|          | 17.1kB /  332MB            

  ...eckpoint-120/optimizer.pt:   0%|          | 90.9kB /  665MB            

  ...eckpoint-120/scheduler.pt: 100%|##########| 1.47kB / 1.47kB            

  ...kpoint-120/tokenizer.json:  73%|#######3  | 25.2MB / 34.4MB            

  ...int-120/training_args.bin:   1%|          |  61.0B / 6.29kB            

CommitInfo(commit_url='https://huggingface.co/sayedyounes/stack_exc_multilabel_base_lm_head/commit/68afcec99d2d8c4fa4b2235dd332fec37b05b4a6', commit_message='Upload folder using huggingface_hub', commit_description='', oid='68afcec99d2d8c4fa4b2235dd332fec37b05b4a6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sayedyounes/stack_exc_multilabel_base_lm_head', endpoint='https://huggingface.co', repo_type='model', repo_id='sayedyounes/stack_exc_multilabel_base_lm_head'), pr_revision=None, pr_num=None)

In [70]:
wandb.finish()

eval/entropy,█▄▄▁▃▂
eval/loss,█▅▃▃▁▁
eval/mean_token_accuracy,▁▃▄▇██
eval/num_tokens,▁▂▄▅▇█
eval/runtime,█▁▇█▅▆
eval/samples_per_second,▁█▂▁▄▃
eval/steps_per_second,▁█▂▁▄▃
train/entropy,█▅▃▂▁▂
train/epoch,▁▁▂▂▄▄▅▅▆▆███
train/global_step,▁▁▂▂▄▄▅▅▆▆███
+5,...
